# Hackathon: Credit Lending

You work at a Bank in the credit risk department. Your stakeholders want to be able to determine the level of risk for all internal customers, so that if the customer applies for a new product (eg. loan, credit card, overdraft, mortgage) the correct decision can be made as to whether they should get the product or not.

The problem with the curent set up is that each individual product has their own way of determining a customer's level of risk. This means that a customer may be rejected to extend their overdraft to €2.000 but accepted to have a loan of €2.000 at the same time. 

This is providing a bad customer journey which is why the bank wants to consolidate this view of risk so that consistent lending decisions can be made.

Often customers who get rejected will complain about their decision so it is important that we can send advisers to them to talk through their situation and suggest a good option for their lending.

**Note:** The regulators consider customers are "unable to repay" when they have missed 3 payments (on any product) in the last 2 years. 

<img src="images/credit.jpeg" style="display: block;margin-left: auto;margin-right: auto;height: 200px"/>


## About the data

We have access to data on a large number of customers.

The unable to repay flag is named `bad_flag` and is determined using the description above. The rest of the features in the dataset are described below:

|Variable Name|	Description	|Type|
|:---|:---|:---|
|rev_unsecured|	Total balance on loans (no real estate) as a percentage of the total loan taken 	|percentage|
|age	|Age of borrower in years	|integer|
|days_past_due_30	|Number of times borrower has been 30-59 days past due in the last |2 years |integer|
|debt_ratio	|Monthly debt payments, alimony, living costs divided by monthy gross income|	float (%)|
|income |	Monthly income	|float|
|num_credit	|Number of Open loans (car loan, mortgage, credit cards etc.)	|integer|
|num_days_late	|Number of times borrower has been 90 days or more past due.	|integer|
|num_realestate	|Number of mortgage and real estate loans including home equity lines of credit	|integer|
|days_past_due_60	|Number of times borrower has been 60-89 days past due in the last 2 years.|	integer|
|num_deps|	Number of dependents in family excluding themselves (spouse, children etc.)	|integer|
|bad_flag	|Missed 3 monthly payments or more in past two years |	Y/N|


## Initial Data Exploration

In [ ]:
import pandas as pd

In [ ]:
credit = pd.read_csv('data/credit.csv')
credit.head()

1. How could you show the datatype of each column? 

In [ ]:
credit.columns

2. Use `.value_counts()` to find out how many customers have no dependents.

3. Use groupby to find the mean value for all the features when split (groupby) the `bad_flag`

4. What is the bad rate (mean of the `bad_flag` feature) when the data is grouped by age? Bonus: Make a plot to show this - are there any outliers?

Hint: `df.groupby('col_to_group')['col_to_agg'].mean().sort_index().plot()`

5. What is the average income by age? Bonus: Make a plot to show this.

**Answers** Data Exploration

In [ ]:
# %load answers/data-exploration.py

# Prepare `X` and `y`

Split the data into `X` and `y` where `X` is the feature matrix and `y` is the target (`bad_flag`)

Check the shape of `X` and `y`. You should see `(150000, 10)` and `(150000,)`

Perform a `value_counts()` on `y` to see how many `bad_flags` of `1` there are. Use `normalize=True` to see this as a percentage.

**Answers** Prepare X and y

In [ ]:
# %load answers/prepare-x-y.py

## Train Test Split

Perform the train test split on the data to create `X_train`, `X_test`, `y_train`, `y_test`

There are not many customers who end up as `bad` customers, therefore use `stratify=y` to ensure the same proportion of bads are in the train and test.

Use a `random_state` to ensure the split is the same each time it is run.

Check the shape of `X_train`, `X_test`, `y_train` and `y_test`

Use `series.value_counts(normalize=True)` on both y_train and y_test to check that the proportion of bads remained the same for each sample.

**Answers** Train test split

In [ ]:
# %load answers/train-test.py

## Preprocessing

Check to see if there are any categorical features.

Now check to see if there is any missing data.

There are missing values in `income` and `num_deps`.

Let's do a little analysis on both these columns. We will start with `num_deps` since there are a small number of unique values in this column (compared to `income`).

- Using the original data `credit`, groupby `num_deps` and find the `mean` of the `bad_flag`
- Pass `dropna=False` into the groupby method to keep the missing values for comparison
- ***Separately*** find the mean of the `num_deps` column - would imputing using the mean or a constant value make sense here?


Syntax:
```python
df
.groupby('col_name', dropna=False)['col_to_agg'].mean()
.sort_index()
```

When looking at the mean value of `num_deps` and comparing to the actual values and their bad rate it's clear that the mean is not a good strategy for imputing missing values in `num_deps`. It would make more sense to impute with a constant value of `0`, since that has the most similar bad rate. This is also the `most_frequent` value.

**Now let's look into the income column**

Find out the bad rate for the missing values of income and compare to the rest of the data:
- Create a new column `income_round` equal to the `income` rounded to the nearest 1,000 
- Groupby this new column passing in the parameter `dropna=False` so you can compare the missings
- Find the mean of the `bad_flag` to get the bad rate
- Sort by the index to make it easier to read
- ***Separately*** find the mean of the income column - would imputing using the mean or a constant value make sense here?

Syntax:
```python
df
.assign(new_col = lambda df: df['col'].round(-3))
```

The bad rate for the missing income is `5.61%` which is closest to the value `income_rounded` of 7000, which in turn is the same place the mean income would sit. Therefore choosing the mean which is very close to this is a good strategy.

**Answers** Preprocessing

In [ ]:
# %load answers/preprocessing.py

## Column Transformer

Import`ColumnTransformer` from `sklearn.compose`

Let's build a `ColumnTransformer()` with two steps:

- `SimpleImputer(strategy='mean')` on the `income` column
- `SimpleImputer(strategy='most_frequent')` on the `num_deps` column

Syntax, change `PreprocessingTransformer` for `SimpleImputer()`:
```python
# Build the transformer
column_imputer = ColumnTransformer(
    [
        ('transformer1', PreprocessingTransformer, ['col1']),
        ('transformer2', PreprocessingTransformer, ['col2'])
    ]
    , remainder='passthrough'
)

# Try it out on the data
pd.DataFrame(column_imputer.fit_transform(X_train), columns=columns=column_imputer.get_feature_names_out())
```

**Answers** Column Transformer

In [ ]:
# %load answers/column-transformer.py

## Building the Model

Now that we have a ColumnTransformer ready to go it's time to build a pipeline.

Since we want the model to be interpretable, a DecisionTree is a good place to start. 

- Import `Pipeline` from `sklearn.pipeline` and `DecisionTreeClassifier` from `sklearn.tree`.
- Instantiate the decision tree model with a max_depth of `10`. Make sure to set a `random_state` in your DecisionTreeClassifier.

Fit the pipeline to `X_train` and `y_train`

**Answers** Building the Model

In [ ]:
# %load answers/build-model.py

## Performance metrics

Find the accuracy on both the train and the test. Is your model generalising well? Is your model overfitting?

The high accuracy score demonstrates a model that generalises well. Since the drop from train to test is minor, this also suggests that the model is not overfitting.

**Answers** Performance metrics

In [ ]:
# %load answers/performance-metrics.py

## Feature importances

Using the pipeline, access the model and then select the feature importances using 
```python
importances = pipeline['model'].feature_importances_
```

Convert this to a pandas series and then plot the data in a bar plot using 
```python
pd.Series(importances, index=features).sort_values().plot(kind='barh')
```

Re-build the model using fewer features. Does this impact performance at all?

- Reduce to 5 features
- Change the max_depth of the DecisionTreeClassifier to 4
- Remake your column transformer if needed.

**Answers** Feature Importances

In [ ]:
# %load answers/feature-importances.py

# Visualise your decision tree

Use the function `plot_tree` from `sklearn.tree` to visualise your decision tree with fewer features. Import matplotlib to change the size of the plot:

```python
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5,5)) #change size here
plot_tree(model, 
          feature_names = list_of_feature_columns, 
          filled=True,
          fontsize=10,
          ax=ax,);
```

**Answers** Visualise your decision tree

In [ ]:
# %load answers/visual-tree.py

## Hyperparameter Tuning

Use `GridSearchCV()` to search across the different model parameters for a decision tree.  You can use either pipeline for this part of the hackathon.

- Check out the actual names of the model parameters using `pipeline.get_params()`
- Create a parameters dictionary listing all the different parameters to try
- Make sure to instantiate the grid. Eg. `grid = GridSearchCV(pipeline, parameters)`

What were the best parameters according to the grid search and what was the average accuracy score?

Use `grid.score(X, y)` to find the accuracy score on your train and test data.

Make a dataframe from the `cv_results_` from your `grid` and `sort_values()` by `rank_test_score` to see the best models and their parameters.

**Answers** Hyperparameter Tuning

In [ ]:
# %load answers/hyperparameter-tuning.py

<img src='images/gdd-logo.png' align=right width=300px>

# Conclusion

You have now gone through a model build from start to finish following these steps:

- Data Exploration
- Split into X and y
- Using train_test_split
- Preprocessing and Column Transformer
- Building the Model
- Performance Metrics
- Feature Importances
- Visualising the Decision Tree
- Hyperparameter Tuning

This is a great place to be in to start your first Machine Learning problems!

## Next Steps

There are more advanced techniques that you might also want to implement. Your next topic of learning may be one of the following

- Further metrics and how to interpret them
- Engineering and selecting features
- Building your own sci-kit learn estimator
- Model interpretation

All of these are visited in our Advanced Data Science with Python course!